In [1]:
import sqlite3
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 80)

conn = sqlite3.connect('../industrial_cluster.db')

# List tables in the database
tables_query = "SELECT name FROM sqlite_master WHERE type='table';"
tables = pd.read_sql_query(tables_query, conn)
print("Tables in the database:")
print(tables)

Tables in the database:
                 name
0           companies
1          components
2             streams
3  stream_composition
4               flows


## companies

In [2]:
companies = pd.read_sql('SELECT * FROM companies ORDER BY company_id', conn)
print(f'{len(companies)} rows')
companies

15 rows


,company_id,name,sector,location,scaling_factor,included,normalize_stream_id
0,C001,Hydrogen Plant,None,None,1.0,1,S006
1,C002,Natural Gas CHP,None,None,1.0,1,NaN
2,C003,Olefins Production Plant,None,None,1.0,1,NaN
3,C004,Aromatics Production Plant,None,None,1.0,1,NaN
4,C005,Biogas Plant,None,None,1.0,1,NaN
5,C006,Ethylene Oxide-Ethylene Glycol,None,None,1.0,1,NaN
6,C007,Biodiesel Production,None,None,1.0,1,NaN
7,C008,PET Production Plant,None,None,1.0,0,NaN
8,C009,Cyclohexane Production Plant,None,None,1.0,1,NaN
9,C010,Methanol Production Plant,None,None,1.0,1,NaN


## components

In [3]:
components = pd.read_sql('SELECT * FROM components ORDER BY component_id', conn)
print(f'{len(components)} rows  |  needs_review=1: {components.needs_review.sum()}')
components

237 rows  |  needs_review=1: 8


,component_id,name,aliases,category,cas_number,molecular_weight,carbon_atoms,hazardous,needs_review,notes,carbon_weight_pct,carbon_weight_pct_manual
0,CM001,"1,1,1,2-Tetrachloroethane",TECE,NaN,630-20-6,167.85,2.0,None,0,NaN,0.143116,None
1,CM002,"1,1,1,2-Tetrafluoroethane",TEFE,NaN,811-97-2,102.03,2.0,None,0,NaN,0.235441,None
2,CM003,"1,1,2-Trichloroethane",TCE,NaN,79-00-5,133.40,2.0,None,0,NaN,0.180075,None
3,CM004,"1,2,3-Trichloropropane",TCP,NaN,96-18-4,147.43,3.0,None,0,NaN,0.244408,None
4,CM005,"1,2,4-Trimethylbenzene","TMB, 4-Trimethylbenzene",NaN,95-63-6,120.19,9.0,None,0,NaN,0.899401,None
...,...,...,...,...,...,...,...,...,...,...,...,...
232,CM255,food waste,NaN,named_material,NaN,NaN,NaN,None,0,NaN,NaN,None
233,CM258,METHY-01,NaN,NaN,NaN,NaN,NaN,None,1,"Process simulation code, likely a fatty acid methyl ester (oleate/linoleate/...",NaN,None
234,CM259,METHY-02,NaN,NaN,NaN,NaN,NaN,None,1,"Process simulation code, likely a fatty acid methyl ester. See METHY-01 notes.",NaN,None
235,CM260,METHY-03,NaN,NaN,NaN,NaN,NaN,None,1,"Process simulation code, likely a fatty acid methyl ester. See METHY-01 notes.",NaN,None


### Components flagged for review

In [4]:
components[components.needs_review == 1][['component_id', 'name', 'aliases', 'category']]

,component_id,name,aliases,category
227,CM231,3-methyl heptane 5% o-xylene,NaN,NaN
228,CM232,butene,NaN,NaN
229,CM234,BDI,NaN,NaN
231,CM249,Pyridine/Pyrrole,NaN,NaN
233,CM258,METHY-01,NaN,NaN
234,CM259,METHY-02,NaN,NaN
235,CM260,METHY-03,NaN,NaN
236,CM264,METFORM,NaN,NaN


## streams

In [5]:
streams = pd.read_sql('SELECT * FROM streams ORDER BY stream_id', conn)
print(f'{len(streams)} rows')
streams

169 rows


,stream_id,company_id,stream_name,stream_type,direction,flow_kton_per_year,temperature_c,pressure_bar,composition_raw,notes,norm_flow_kton_per_year,carbon_pct,carbon_pct_complete
0,S001,C001,FS-NG,raw_material,input,3.342041,20.00,20.00,"70.5% CH4, 2.38% CO2, 21.78 N2, 4.88% N-C2A, 0.47% NC3-A",None,3.342041,0.577235,1.0
1,S002,C001,FS-STEAM,raw_material,input,6.614592,215.00,21.00,100% water,None,6.614592,NaN,0.0
2,S003,C001,FS-AIR,raw_material,input,27.094167,15.00,1.02,"0.045% CO2, 0.27% H2O, 75.27 % N2, 23.13% O2, 1.28% AR",None,27.094167,0.000123,0.0
3,S004,C001,FS-UCH4,raw_material,input,1.099412,15.00,1.02,"70.5% CH4, 2.38% CO2, 21.78 N2, 4.88% N-C2A, 0.47% NC3-A",None,1.099412,0.577235,1.0
4,S005,C001,FS-NG,raw_material,input,3.342041,20.00,20.00,"70.5% CH4, 2.38% CO2, 21.78 N2, 4.88% N-C2A, 0.47% NC3-A",None,3.342041,0.577235,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
164,S165,C014,WS-PURG2,waste,output,228.080000,40.00,2.00,"H2O (85.3%), UREA (10.2%), NH3 (4.1%), CO2 (0.4%)",None,NaN,0.021490,0.0
165,S166,C014,WS-PURG3,waste,output,8.060000,40.00,2.00,"H2O (52.3%), AMBIC (39%), NH3 (8.7%)",None,NaN,0.180003,0.0
166,S167,C014,WS-WW1,waste,output,45.886500,40.00,1.02,"H2O (93.6%), UREA (6.3%), NH3 (0.1%)",None,NaN,0.012599,0.0
167,S168,C014,WS-WW2,waste,output,90.053000,41.03,1.02,"H2O (93.5%), NH3 (6.5%)",None,NaN,0.000000,0.0


## stream_composition

In [6]:
stream_composition = pd.read_sql("""
    SELECT sc.composition_id, sc.stream_id, s.stream_name, s.company_id,
           c.name AS component, sc.fraction, sc.is_trace
    FROM stream_composition sc
    JOIN streams s    ON sc.stream_id    = s.stream_id
    JOIN components c ON sc.component_id = c.component_id
    ORDER BY sc.stream_id, sc.fraction DESC
""", conn)
print(f'{len(stream_composition)} rows')
stream_composition

583 rows


,composition_id,stream_id,stream_name,company_id,component,fraction,is_trace
0,CP001,S001,FS-NG,C001,Methane,0.7050,0
1,CP005,S001,FS-NG,C001,unknown,0.2177,0
2,CP003,S001,FS-NG,C001,Ethane,0.0488,0
3,CP002,S001,FS-NG,C001,Carbon dioxide,0.0238,0
4,CP004,S001,FS-NG,C001,Propane,0.0047,0
...,...,...,...,...,...,...,...
578,CP579,S168,WS-WW2,C014,Water,0.9350,0
579,CP580,S168,WS-WW2,C014,Ammonia,0.0650,0
580,CP581,S169,WS-WW3,C014,Water,0.9830,0
581,CP582,S169,WS-WW3,C014,Urea,0.0140,0


## flows

In [7]:
flows = pd.read_sql('SELECT * FROM flows ORDER BY flow_id', conn)
print(f'{len(flows)} rows  (populated after matching analysis)')
flows

0 rows  (populated after matching analysis)


,flow_id,from_company_id,to_company_id,from_stream_id,to_stream_id,flow_kton_per_year,status,notes


# Carbon content calculation


In [8]:
# List all companies and their names
company_names = pd.read_sql('SELECT company_id, name FROM companies ORDER BY company_id', conn)
print(f'{len(company_names)} rows')
# display(company_names)

# Query all streams and company information in one table. But only the methanol and dme plants for now
stream_company = pd.read_sql("""
    SELECT 
        s.stream_id, 
        s.stream_name, 
        s.company_id, 
        c.name,
        s.flow_kton_per_year,
        s.norm_flow_kton_per_year,
        s.stream_type,
                             s.carbon_pct,
                             s.carbon_pct_complete,
                             s.direction
    FROM streams s
    JOIN companies c ON s.company_id = c.company_id
    WHERE c.company_id IN ('C013', 'C010')
    ORDER BY s.stream_id
""", conn)
print(f'{len(stream_company)} rows')
stream_company

15 rows
17 rows


,stream_id,stream_name,company_id,name,flow_kton_per_year,norm_flow_kton_per_year,stream_type,carbon_pct,carbon_pct_complete,direction
0,S125,FS-AIR,C010,Methanol Production Plant,827.760128,None,raw_material,0.000000,1,input
1,S126,FS-BFW1,C010,Methanol Production Plant,80.204490,None,raw_material,NaN,0,input
2,S127,FS-Methane,C010,Methanol Production Plant,50.000000,None,raw_material,0.748815,1,input
3,S128,FS-Methane-2,C010,Methanol Production Plant,32.220319,None,raw_material,0.748815,1,input
4,S129,FS-CO2,C010,Methanol Production Plant,38.817403,None,raw_material,0.272915,1,input
5,S130,FS-WATER,C010,Methanol Production Plant,157.213547,None,raw_material,NaN,0,input
6,S131,PS-CH3OH,C010,Methanol Production Plant,93.716777,None,products,0.374829,0,output
7,S132,PS-MPS,C010,Methanol Production Plant,80.204490,None,products,NaN,0,output
8,S133,WS-FG,C010,Methanol Production Plant,859.980447,None,waste,0.054189,0,output
9,S134,WS-LIGHT,C010,Methanol Production Plant,7.094102,None,waste,0.274797,0,output


In [9]:
# List the composition of the S134 stream (look at the specification file for table details)
s134_composition = pd.read_sql("""
    SELECT sc.composition_id, c.component_id,sc.stream_id, s.stream_name, s.company_id,
           c.name AS component, sc.fraction, sc.is_trace, c.carbon_weight_pct, c.carbon_atoms
    FROM stream_composition sc
    JOIN streams s    ON sc.stream_id    = s.stream_id
    JOIN components c ON sc.component_id = c.component_id
    WHERE s.stream_id = 'S134'
    ORDER BY sc.fraction DESC
""", conn)
print(f'{len(s134_composition)} rows')
s134_composition

7 rows


,composition_id,component_id,stream_id,stream_name,company_id,component,fraction,is_trace,carbon_weight_pct,carbon_atoms
0,CP478,CM055,S134,WS-LIGHT,C010,Carbon dioxide,0.7994,0,0.272915,1.0
1,CP479,CM112,S134,WS-LIGHT,C010,Methanol,0.1335,0,0.374875,1.0
2,CP481,CM227,S134,WS-LIGHT,C010,unknown,0.0348,0,NaN,NaN
3,CP477,CM056,S134,WS-LIGHT,C010,Carbon monoxide,0.0154,0,0.427438,1.0
4,CP476,CM094,S134,WS-LIGHT,C010,Hydrogen,0.0130,0,0.000000,0.0
5,CP480,CM264,S134,WS-LIGHT,C010,METFORM,0.0035,0,NaN,NaN
6,CP475,CM225,S134,WS-LIGHT,C010,Water,0.0004,0,NaN,0.0


In [11]:
# select all from CP481 component
cp481 = pd.read_sql("""
    SELECT *
    FROM components
    WHERE component_id = 'CM264'
""", conn)
print(f'{len(cp481)} rows')
cp481

1 rows


,component_id,name,aliases,category,cas_number,molecular_weight,carbon_atoms,hazardous,needs_review,notes,carbon_weight_pct,carbon_weight_pct_manual
0,CM264,METFORM,None,None,None,None,None,None,1,"Likely methyl formate (CM118, alias MF) — a common byproduct in methanol syn...",None,None
